# 🧠 Fine-Tuning Models: Freezing Strategies and Learning Rates

Welcome to the hands-on explanation notebook for **Fine-Tuning Models**! In this notebook, we will:
1. Distinguish between Transfer Learning and Fine-Tuning.
2. Outline the workflow: head swapping, selective layer freezing, and low learning rates.
3. Load a `ResNet18` model and implement a flexible **name-based layer freezing utility** in PyTorch.
4. Compare three fine-tuning strategies:
   - **Full Fine-Tuning:** All layers trainable.
   - **Head-Only Fine-Tuning:** All backbone layers frozen.
   - **Partial Fine-Tuning:** Early layers frozen, deep layers and head trainable.
5. Contrast the trainable parameters and computational load of each strategy.
6. Connect these strategies to YOLO's `freeze` and `lr0` configurations.

Let's start by importing the necessary libraries.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# Set seed for reproducibility
torch.manual_seed(42)

## 1. Implementing the Layer Freezing Utility

We implement a utility that freezes layers in a PyTorch model based on name substrings.

In [ ]:
def freeze_layers_by_name(model, match_names):
    """
    Freeze parameters if their layer name contains any of the strings in match_names.
    """
    for name, param in model.named_parameters():
        if any(match_str in name for match_str in match_names):
            param.requires_grad = False
        else:
            param.requires_grad = True

def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

## 2. Comparing Fine-Tuning Strategies

Let's evaluate the trainable parameters for three popular strategies:

### Strategy A: Full Fine-Tuning (Train Everything)
We keep all layers unfrozen but train with a lower learning rate.

In [ ]:
model_full = models.resnet18()
model_full.fc = nn.Linear(model_full.fc.in_features, 10)

total_p, trainable_p = count_parameters(model_full)
print("--- Strategy A: Full Fine-Tuning ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

### Strategy B: Head-Only Fine-Tuning (Freeze Feature Extractor)
We freeze everything except the final classification layer (`fc`).

In [ ]:
model_head = models.resnet18()
model_head.fc = nn.Linear(model_head.fc.in_features, 10)

freeze_layers_by_name(model_head, match_names=['conv', 'bn', 'layer'])

total_p, trainable_p = count_parameters(model_head)
print("--- Strategy B: Head-Only Fine-Tuning ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

### Strategy C: Partial Fine-Tuning (Freeze Early, Train Deep)
Early layers (conv1, bn1, layer1, layer2) detect generic features and are frozen. Deep layers (layer3, layer4) and the head are kept trainable to adapt to custom objects.

In [ ]:
model_partial = models.resnet18()
model_partial.fc = nn.Linear(model_partial.fc.in_features, 10)

freeze_layers_by_name(model_partial, match_names=['conv1', 'bn1', 'layer1', 'layer2'])

total_p, trainable_p = count_parameters(model_partial)
print("--- Strategy C: Partial Fine-Tuning ---")
print(f"Total Parameters    : {total_p:,}")
print(f"Trainable Parameters: {trainable_p:,}")

Observe:
-   **Strategy A (Full):** $11.2$ Million trainable parameters. Highly flexible, but high risk of **Catastrophic Forgetting** if learning rate is too large.
-   **Strategy B (Head-only):** Only $5,130$ trainable parameters. High speed, low overfitting, but can underfit if target images differ significantly from COCO.
-   **Strategy C (Partial):** $9.4$ Million trainable parameters. Preserves low-level edge features while optimizing deep layers to detect custom object shapes. This is the optimal strategy for mid-sized datasets.

## 💡 Connection to YOLO and Deep Learning
*   **The `freeze` Argument:** In YOLO, setting `freeze=10` freezes the first 10 layers of the model. 
*   YOLO's architecture features a backbone (early representation extraction) and a neck/head (multi-scale feature integration and bounding box classification). Freezing the backbone ensures pre-trained spatial parameters are conserved, which is essential for training with small target datasets.
*   **Lower Learning Rate (`lr0`):** Standard training from scratch uses `lr0=0.01`. Fine-tuning typically reduces this to `lr0=0.001` or `lr0=0.0001` to prevent destroying pre-trained filters.